# Flat CSV sessions: stimulus-triggered running and spikes

For runs logged by the older Bonsai workflows (`DotMotion_SpeedTuning`, `DriftingGratings_Direction`, ...). These
write each stream as its own CSV file in one folder instead of the aeon / open-rigs layout:

```
M26031_DriftingGratings_Direction_PDAsync2026-10-08T16_00_40.csv   photodiode and sync pulse state
M26031_DriftingGratings_Direction_Trial2026-10-08T16_00_40.csv     one row per trial: its parameters
M26031_DriftingGratings_Direction_Video2026-10-08T16_00_40.csv     camera frames and LED sync
M26031_DriftingGratings_Direction_WheelLog2026-10-08T16_00_40.csv  wheel encoder and last sync pulse
```

`open_rigs_hf_visual.bonsai_csv` reads them and puts them on the NI-DAQ clock. After that, the same general
functions as in `psth.ipynb` are used (`traces`, `spikes`). All times are seconds on the NI-DAQ clock.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import swc.aeon.io.api as aeon

from open_rigs_hf_visual import spikes as sp
from open_rigs_hf_visual import traces as tr
from open_rigs_hf_visual.bonsai_csv import find_runs, load_csv_session, trial_table
from open_rigs_hf_visual.hf_visual import running_speed

## Runs in the folder

One folder can hold several protocols, and repeats of one protocol. Each run is identified by its protocol and start
time.

In [ ]:
folder = Path(r"Z:\ibn-vision\DATA\SUBJECTS\M26031\stimuli\20261008\M26031_20261008_1")
runs = find_runs(folder)
runs.drop_duplicates(["protocol", "start"])[["protocol", "start"]]

## Session

Pick the run (`protocol`, and `start` if it was run more than once: a time, or `-1` for the last) and the SpikeGLX
folder recorded during it. Each `_gN` folder is one SpikeGLX run. Pick the one whose start time (`fileCreateTime` in
the `.nidq.meta` file) is just before the run's start.

`load_csv_session` aligns the Arduino clock to the NI-DAQ by matching sync pulses (`LastSyncPulseTime` in the wheel
log against NI-DAQ `XA0`). The Arduino restarts its pseudo-random pulse sequence each session, so the pulses can
also "match" the wrong recording. The loader therefore checks that Bonsai and SpikeGLX clock times agree after
alignment, and stops with an error if they differ by more than a minute.

In [ ]:
protocol = "DriftingGratings"
start = None  # e.g. -1 or "2026-10-08T15:36:36" when the protocol was run more than once
ephys_root = Path(r"Z:\ibn-vision\DATA\SUBJECTS\M26031\ephys\20261008\20261008_1\M26031_20261008_1_g1")
kilosort_dir = None  # spike sorting output, once it exists

session = load_csv_session(folder, ephys_root, protocol=protocol, start=start, kilosort_dir=kilosort_dir)
print(session.protocol, session.start)
print("Arduino -> NI-DAQ:", session.arduino_to_nidq)
print(f"Bonsai vs SpikeGLX clock after alignment: {session.clock_offset:+.2f} s")
print("streams:", {name: len(data) for name, data in session.streams.items()})

## Trials

`trial_table` gives one row per logged trial. Its onset is the photodiode rising edge on the NI-DAQ
(`photodiode_channel`) nearest the logged time, and its offset is the next falling edge.

Set `condition` below to the trial parameter that defines the stimulus condition. Examples are included for the two
protocols. Blank trials (contrast 0) are labelled `"blank"`.

In [ ]:
photodiode_channel = "XA3"
trials = trial_table(session, photodiode_channel=photodiode_channel)
print(f"{trials['matched'].sum()} of {len(trials)} onsets matched to a photodiode edge")
print(f"logged onsets lead the photodiode by {np.median(trials['onset'] - trials['onset_logged']) * 1000:.0f} ms (median)")
print(f"photodiode-high duration: {np.nanmedian(trials['duration']):.3f} s")

if "angle_g1" in trials:  # drifting gratings
    # zero-padded so conditions sort by angle ("000", "045", ... "315", then "blank")
    trials["condition"] = trials["angle_g1"].map("{:03d}".format).where(trials["contrast_g1"] > 0, "blank")
elif "vel_x1" in trials:  # dot motion: dot speed
    trials["condition"] = trials["vel_x1"]
else:
    trials["condition"] = "all"
trials.groupby("condition").size()

## Running speed

Speed is the derivative of the cumulative wheel count (`wheel` column). Without a wheel calibration it is in encoder
counts/s. Set `cm_per_count` (wheel circumference / counts per revolution) to get cm/s.

In [ ]:
cm_per_count = None
speed_units = "cm/s" if cm_per_count else "counts/s"
speed = running_speed(session.wheel, column="wheel", cm_per_count=cm_per_count, smooth=0.05)

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.plot(aeon.to_seconds(speed.index), speed, lw=0.5, color="k")
ax.plot(trials["onset"], np.zeros(len(trials)), "|", color="C3", ms=6, label="trial onsets")
ax.set(xlabel="NI-DAQ time (s)", ylabel=f"speed ({speed_units})", title="Running speed over the run")
ax.legend(loc="upper right")
plt.tight_layout()

## Stimulus-triggered running speed

The mean speed (± SEM across trials) around every trial onset, then the same for each condition. The shaded span
is the median photodiode-high period.

In [ ]:
window = (-0.5, 1.5)
stim_end = float(np.nanmedian(trials["duration"]))
aligned_speed = tr.align_trace(speed, trials["onset"], window, dt=0.01)
by_condition = tr.align_trace_by_condition(speed, trials["onset"], trials["condition"], window, dt=0.01)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
tr.plot_mean(axes[0], aligned_speed, color="k")
axes[0].set(title=f"All trials (n={aligned_speed.n_trials})", ylabel=f"speed ({speed_units})")
cmap = plt.get_cmap("twilight", max(len(by_condition), 2))
for i, (label, aligned) in enumerate(by_condition.items()):
    color = "k" if label == "blank" else cmap(i)
    tr.plot_mean(axes[1], aligned, label=f"{label} (n={aligned.n_trials})", color=color, show_sem=False)
axes[1].set(title="By condition")
axes[1].legend(fontsize="x-small", ncol=2)
for ax in axes:
    ax.axvspan(0, stim_end, color="0.9", zorder=0)
    ax.axvline(0, color="k", lw=0.5)
plt.tight_layout()

## Trial-by-trial variation

Each row of the heatmap is one trial. Trials count as running when the mean speed in the 0.5 s before onset is above
`running_threshold`.

In [ ]:
running_threshold = 10.0  # counts/s (cm/s if cm_per_count is set)
trials["speed_pre"] = aligned_speed.trial_mean(window[0], 0.0)
trials["speed_stim"] = aligned_speed.trial_mean(0.0, stim_end)
trials["speed_change"] = trials["speed_stim"] - trials["speed_pre"]
trials["running"] = trials["speed_pre"] > running_threshold
print(f"{trials['running'].mean():.0%} of trials start while running")

fig, axes = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw={"width_ratios": [3, 1, 2]})
vmax = np.nanpercentile(aligned_speed.values, 99)
image = tr.plot_trials(axes[0], aligned_speed, vmin=0, vmax=vmax)
axes[0].axvline(0, color="w", lw=0.5)
axes[0].set(title="Speed around each trial")
fig.colorbar(image, ax=axes[0], label=f"speed ({speed_units})")
axes[1].plot(trials["speed_pre"], trials["trial"], ".", ms=2, color="k")
axes[1].axvline(running_threshold, color="C3", lw=0.8)
axes[1].set(ylim=(len(trials) - 0.5, -0.5), xlabel=f"pre-onset ({speed_units})", title="Per trial")
for state, color in ((False, "C0"), (True, "C3")):
    subset = trials["running"] == state
    if subset.any():
        label = f"{'running' if state else 'stationary'} (n={subset.sum()})"
        tr.plot_mean(axes[2], tr.align_trace(speed, trials.loc[subset, "onset"], window), label=label, color=color)
axes[2].axvspan(0, stim_end, color="0.9", zorder=0)
axes[2].set(ylabel=f"speed ({speed_units})", title="Split by state at onset")
axes[2].legend(fontsize="small")
plt.tight_layout()

In [ ]:
summary = trials.groupby("condition").agg(
    n=("trial", "size"),
    speed_pre=("speed_pre", "mean"),
    speed_stim=("speed_stim", "mean"),
    change=("speed_change", "mean"),
    change_sem=("speed_change", "sem"),
    running_fraction=("running", "mean"),
)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(summary)), summary["change"], yerr=summary["change_sem"], color="0.4", capsize=2)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(range(len(summary)), summary.index)
ax.set(xlabel="condition", ylabel=f"speed change ({speed_units})", title="During minus before onset")
plt.tight_layout()
summary.round(2)

## Spikes

Needs spike sorting (`kilosort_dir` in the Session cell). Each figure shows one unit's raster and PSTH, grouped by
condition.

In [ ]:
if session.spikes is None:
    raise RuntimeError("No spike sorting loaded: set kilosort_dir in the Session cell to run the cells below.")
unit_ids = session.spikes.unit_ids(label="good")
print(f"{len(unit_ids)} good units")
for unit in unit_ids[:5]:
    sp.plot_unit(
        session.spikes.spikes(unit), trials["onset"], trials["condition"], window=window,
        bin_size=0.02, smooth=0.02, shade=(0, stim_end), title=f"unit {unit}",
    )